In [19]:
import pandas as pd
import numpy as np

# Đường dẫn chính xác đã được xác thực từ kết quả quét file
train_path = '/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv'
test_path = '/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv'

# Đọc dữ liệu
train_hp = pd.read_csv(train_path)
test_hp = pd.read_csv(test_path)

print("Kích thước tập Train giá nhà:", train_hp.shape)
print("Kích thước tập Test giá nhà:", test_hp.shape)

# Kiểm tra các cột bị thiếu dữ liệu 
missing_count = train_hp.isnull().sum()
missing_percent = (train_hp.isnull().sum() / len(train_hp)) * 100

# Tạo bảng tổng hợp các cột bị thiếu dữ liệu
missing_df = pd.DataFrame({
    'Số lượng thiếu': missing_count,
    'Tỷ lệ phần trăm (%)': missing_percent
})

# Lọc ra các cột thực sự có giá trị thiếu và sắp xếp theo số lượng giảm dần
missing_df = missing_df[missing_df['Số lượng thiếu'] > 0].sort_values(by='Số lượng thiếu', ascending=False)

print("\nTop các cột bị thiếu dữ liệu trong tập Train:")
display(missing_df.head(15))

Kích thước tập Train giá nhà: (1460, 81)
Kích thước tập Test giá nhà: (1459, 80)

Top các cột bị thiếu dữ liệu trong tập Train:


,Số lượng thiếu,Tỷ lệ phần trăm (%)
PoolQC,1453,99.520548
MiscFeature,1406,96.301370
Alley,1369,93.767123
Fence,1179,80.753425
MasVnrType,872,59.726027
FireplaceQu,690,47.260274
LotFrontage,259,17.739726
GarageType,81,5.547945
GarageYrBlt,81,5.547945
GarageFinish,81,5.547945


In [20]:
# 1. Các cột dạng chữ (categorical) mà giá trị thiếu có nghĩa là "Không có" (None)
cols_cat_none = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'MasVnrType'
]

# Điền giá trị 'None' cho các cột này ở cả tập Train và Test
for col in cols_cat_none:
    if col in train_hp.columns:
        train_hp[col] = train_hp[col].fillna('None')
    if col in test_hp.columns:
        test_hp[col] = test_hp[col].fillna('None')

# 2. Các cột dạng số (numerical) liên quan đến Gara hoặc diện tích mà thiếu nghĩa là 0 (Không có)
cols_num_zero = ['GarageYrBlt', 'GarageArea', 'GarageCars', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']

for col in cols_num_zero:
    if col in train_hp.columns:
        train_hp[col] = train_hp[col].fillna(0)
    if col in test_hp.columns:
        test_hp[col] = test_hp[col].fillna(0)

# 3. Riêng cột LotFrontage (Đường diện tiền nhà) thiếu thì ta điền bằng số trung median của cột
if 'LotFrontage' in train_hp.columns:
    median_lot = train_hp['LotFrontage'].median()
    train_hp['LotFrontage'] = train_hp['LotFrontage'].fillna(median_lot)
    test_hp['LotFrontage'] = test_hp['LotFrontage'].fillna(median_lot)

# Kiểm tra lại xem còn cột nào bị thiếu dữ liệu trong tập train không
con_thieu_train = train_hp.isnull().sum().sum()
con_thieu_test = test_hp.isnull().sum().sum()

print(f"Tổng số ô còn thiếu dữ liệu trong tập Train: {con_thieu_train}")
print(f"Tổng số ô còn thiếu dữ liệu trong tập Test: {con_thieu_test}")

Tổng số ô còn thiếu dữ liệu trong tập Train: 1
Tổng số ô còn thiếu dữ liệu trong tập Test: 12


In [21]:
# 1. Điền nốt các cột số còn sót lại bằng số trung vị (median) của tập train
cols_num_train = train_hp.select_dtypes(include=['int64', 'float64']).columns
for col in cols_num_train:
    if train_hp[col].isnull().sum() > 0:
        median_val = train_hp[col].median()
        train_hp[col] = train_hp[col].fillna(median_val)
        
cols_num_test = test_hp.select_dtypes(include=['int64', 'float64']).columns
for col in cols_num_test:
    if test_hp[col].isnull().sum() > 0:
        median_val = train_hp[col].median() # Lấy median từ tập train để điền cho test (tránh rò rỉ dữ liệu)
        test_hp[col] = test_hp[col].fillna(median_val)

# 2. Điền nốt các cột chữ (categorical) còn sót lại bằng giá trị xuất hiện nhiều nhất (mode)
cols_cat_train = train_hp.select_dtypes(include=['object']).columns
for col in cols_cat_train:
    if train_hp[col].isnull().sum() > 0:
        mode_val = train_hp[col].mode()[0]
        train_hp[col] = train_hp[col].fillna(mode_val)

cols_cat_test = test_hp.select_dtypes(include=['object']).columns
for col in cols_cat_test:
    if test_hp[col].isnull().sum() > 0:
        mode_val = train_hp[col].mode()[0] # Lấy mode từ tập train để điền cho test
        test_hp[col] = test_hp[col].fillna(mode_val)

# 3. Kiểm tra lại lần cuối xem còn ô trống nào không
print("Tổng số ô còn thiếu sau khi xử lý triệt để - Train:", train_hp.isnull().sum().sum())
print("Tổng số ô còn thiếu sau khi xử lý triệt để - Test:", test_hp.isnull().sum().sum())

Tổng số ô còn thiếu sau khi xử lý triệt để - Train: 0
Tổng số ô còn thiếu sau khi xử lý triệt để - Test: 0


In [22]:
import pandas as pd
import numpy as np

# 1. Tách biến mục tiêu (SalePrice) và cột Id ra khỏi tập Train
# Biến y_train chính là giá nhà mà chúng ta cần dự đoán
y_train = train_hp['SalePrice']

# X_train và X_test bỏ cột 'Id' vì ID không phải là đặc trưng dự đoán
X_train = train_hp.drop(columns=['SalePrice', 'Id'])
X_test = test_hp.drop(columns=['Id'])

# 2. Gộp chung X_train và X_test lại để mã hóa One-Hot đồng nhất cột
# Việc này giúp đảm bảo tập train và test có cùng số lượng đặc trưng (features) sau khi mã hóa
df_combined = pd.concat([X_train, X_test], axis=0)
df_combined_encoded = pd.get_dummies(df_combined)

# 3. Tách trở lại thành X_train_final và X_test_final
X_train_final = df_combined_encoded.iloc[:len(train_hp)]
X_test_final = df_combined_encoded.iloc[len(train_hp):]

print("Kích thước X_train sau khi mã hóa:", X_train_final.shape)
print("Kích thước X_test sau khi mã hóa:", X_test_final.shape)

Kích thước X_train sau khi mã hóa: (1460, 302)
Kích thước X_test sau khi mã hóa: (1459, 302)


In [23]:
from sklearn.model_selection import train_test_split

# Chia tập train thành 80% học và 20% validation
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_final, 
    y_train, 
    test_size=0.2, 
    random_state=42
)

print("Đã chia dữ liệu thành công!")
print("Kích thước tập huấn luyện (X_tr):", X_tr.shape)
print("Kích thước tập kiểm tra (X_val):", X_val.shape)

Đã chia dữ liệu thành công!
Kích thước tập huấn luyện (X_tr): (1168, 302)
Kích thước tập kiểm tra (X_val): (292, 302)


In [24]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import numpy as np

# 1. Khởi tạo và huấn luyện mô hình Ridge Regression (dạng tuyến tính có điều regularization để chống overfitting)
model_ridge = Ridge(alpha=1.0, random_state=42)

print("Đang huấn luyện mô hình Baseline (Ridge Regression)...")
model_ridge.fit(X_tr, y_tr)
print("Huấn luyện xong!")

# 2. Dự đoán trên tập Validation và tính toán điểm số RMSE
y_pred_val = model_ridge.predict(X_val)
rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))

print(f"\nSai số RMSE trên tập Validation: {rmse_val:.4f}")

Đang huấn luyện mô hình Baseline (Ridge Regression)...
Huấn luyện xong!

Sai số RMSE trên tập Validation: 29737.2316


In [25]:
# 1. Huấn luyện lại trên toàn bộ dữ liệu train
model_ridge.fit(X_train_final, y_train)

# 2. Dự đoán trên tập Test
y_test_pred = model_ridge.predict(X_test_final)

# 3. Tạo file submission
sub_ridge = pd.DataFrame({
    'Id': test_hp['Id'],
    'SalePrice': y_test_pred
})

sub_ridge.to_csv('submission_ridge.csv', index=False)
print("Đã xuất file submission_ridge.csv thành công!")

Đã xuất file submission_ridge.csv thành công!


In [26]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# 1. Khởi tạo mô hình Random Forest Regressor
model_rf = RandomForestRegressor(n_estimators=100, random_state=42)

print("Đang huấn luyện mô hình Random Forest Regressor trên tập Train...")
model_rf.fit(X_tr, y_tr)
print("Huấn luyện xong!")

# 2. Dự đoán trên tập Validation để kiểm tra sai số
y_pred_val_rf = model_rf.predict(X_val)
rmse_val_rf = np.sqrt(mean_squared_error(y_val, y_pred_val_rf))

print(f"\nSai số RMSE của Random Forest trên tập Validation: {rmse_val_rf:.4f}")

Đang huấn luyện mô hình Random Forest Regressor trên tập Train...
Huấn luyện xong!

Sai số RMSE của Random Forest trên tập Validation: 28713.8284


In [27]:
# Huấn luyện lại trên toàn bộ dữ liệu train
print("Đang huấn luyện lại Random Forest trên toàn bộ dữ liệu...")
model_rf.fit(X_train_final, y_train)
print("Huấn luyện toàn bộ xong, mô hình đã sẵn sàng dự đoán tập Test!")

Đang huấn luyện lại Random Forest trên toàn bộ dữ liệu...
Huấn luyện toàn bộ xong, mô hình đã sẵn sàng dự đoán tập Test!


In [28]:
import pandas as pd

# 1. Dự đoán kết quả cho tập Test
print("Đang dự đoán giá nhà cho tập Test bằng Random Forest...")
y_test_pred_rf = model_rf.predict(X_test_final)

# 2. Tạo DataFrame chứa kết quả nộp bài
sub_rf = pd.DataFrame({
    'Id': test_hp['Id'],
    'SalePrice': y_test_pred_rf
})

# 3. Xuất ra file riêng biệt để lưu trữ
sub_rf.to_csv('submission_random_forest.csv', index=False)
print("Đã xuất file submission_random_forest.csv thành công!")

Đang dự đoán giá nhà cho tập Test bằng Random Forest...
Đã xuất file submission_random_forest.csv thành công!
